# Preprocessing Pipeline

Notebook 02 builds the supervised-learning matrices used by later notebooks:

- loads `train.csv` and `test.csv`
- creates the 24-hour closure target from `Created Date` and `Closed Date`
- removes leakage and identifier columns
- handles missing values using parameters learned from train only
- adds temporal, geographic, count-encoding, and interaction features
- encodes remaining categoricals and returns `X_train`, `y_train`, and `X_test`


## Purpose Of This Notebook

The raw NYC 311 table contains dates, categorical fields, missing values, text-like columns, and location fields. It turns those raw fields into model-ready matrices while keeping the same transformations for train and test.

The most important rule is consistency: every transformation learned from training data must be applied to test data in the same way.


In [ ]:
# Data load.
# Resolve the project root before loading files so relative paths stay stable
# whether the notebook is started from the root folder or notebooks/.
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if not (project_root / "data" / "train.csv").exists():
    project_root = project_root.parent

data_dir = project_root / "data"

# Load train and test before preprocessing so the same transformation functions
# can be applied to both datasets.
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

# The raw date strings use this format in the CSV files.
date_format = "%m/%d/%Y %I:%M:%S %p"

# Central settings used by several preprocessing steps.
missing_drop_threshold = 0.90
max_onehot_cardinality = 15
unknown_value = "UNKNOWN"

print("train shape:", train.shape)
print("test shape:", test.shape)


## Target And Leakage Controls

This step creates the binary label and removes fields that would not be available at prediction time. Keeping this boundary clear is essential for a fair model evaluation.


In [ ]:
# 2) Target, temporal features, and leakage removal.
# Create the label, extract safe date features, and drop fields
# that would not be available for future predictions.

def add_target(df):
    """Create target=1 when a request closes between 0 and 24 hours."""
    # Work on a copy so the original raw dataframe stays unchanged.
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    closed = pd.to_datetime(
        df["Closed Date"],
        format=date_format,
        errors="coerce"
    )

    # The target is based on elapsed hours because the project asks whether a
    # complaint was closed within one day.
    hours_to_close = (closed - created).dt.total_seconds() / 3600

    df["target"] = (
        (hours_to_close >= 0)
        & (hours_to_close <= 24)
    ).astype(int)

    return df


def add_temporal_features(df):
    """Extract model-safe features from Created Date."""
    df = df.copy()

    created = pd.to_datetime(
        df["Created Date"],
        format=date_format,
        errors="coerce"
    )

    hour = created.dt.hour

    # These features use only the creation timestamp, which is available at
    # prediction time for both train and test rows.
    df["created_hour"] = hour
    df["created_day_of_week"] = created.dt.dayofweek
    df["created_month"] = created.dt.month
    df["created_is_weekend"] = created.dt.dayofweek.isin([5, 6]).astype(int)

    # Part-of-day turns the numeric hour into a coarse categorical pattern that
    # can capture broad operational differences.
    df["created_part_of_day"] = np.select(
        [
            hour.between(0, 5),
            hour.between(6, 11),
            hour.between(12, 17),
            hour.between(18, 23)
        ],
        [
            "night",
            "morning",
            "afternoon",
            "evening"
        ],
        default=unknown_value
    )

    return df


def convert_known_numeric_columns(df):
    """Convert numeric-looking raw columns into numeric dtype."""
    df = df.copy()

    numeric_cols = [
        "Incident Zip",
        "Council District",
        "BBL",
        "X Coordinate (State Plane)",
        "Y Coordinate (State Plane)",
        "Latitude",
        "Longitude"
    ]

    decimal_comma_cols = {
        "Latitude",
        "Longitude"
    }

    for col in numeric_cols:
        if col in df.columns:
            # Remove spaces and non-breaking spaces before numeric conversion.
            cleaned = (
                df[col]
                .astype("string")
                .str.strip()
                .str.replace("\u00a0", "", regex=False)
                .str.replace(" ", "", regex=False)
            )

            # Coordinates may use a comma as the decimal separator; other numeric
            # fields use commas as thousands separators.
            if col in decimal_comma_cols:
                cleaned = cleaned.str.replace(",", ".", regex=False)
            else:
                cleaned = cleaned.str.replace(",", "", regex=False)

            df[col] = pd.to_numeric(cleaned, errors="coerce").astype(float)

    return df


def remove_leakage_and_ids(df):
    """Drop columns that leak the answer or identify rows too directly."""
    df = df.copy()

    cols_to_drop = [
        "Closed Date",
        "Created Date",
        "Unique Key",
        "Unnamed: 0",
        "Location"
    ]

    existing_cols = [
        col for col in cols_to_drop
        if col in df.columns
    ]

    return df.drop(columns=existing_cols)


# Apply the same cleaning logic to train and test. The target is added only to
# train because test labels are unknown.
train_processed = add_target(train)
train_processed = add_temporal_features(train_processed)
train_processed = convert_known_numeric_columns(train_processed)
train_processed = remove_leakage_and_ids(train_processed)

test_processed = add_temporal_features(test)
test_processed = convert_known_numeric_columns(test_processed)
test_processed = remove_leakage_and_ids(test_processed)

print(train_processed["target"].value_counts())
print(train_processed["target"].value_counts(normalize=True).round(4))

print("train processed shape:", train_processed.shape)
print("test processed shape:", test_processed.shape)


In [ ]:
# 3) Missing values.
# Missing-value rules are fitted on training data and then reused on test data.

def fit_missing_value_params(df):
    """Learn which columns to drop and which medians to use for imputation."""
    missing_fraction = df.isna().mean()

    # Columns with too much missingness are removed instead of imputed
    # aggressively, except the target which must stay in the training table.
    dropped_cols = [
        col
        for col in missing_fraction[
            missing_fraction > missing_drop_threshold
        ].index
        if col != "target"
    ]

    remaining = df.drop(columns=dropped_cols, errors="ignore")

    numeric_data = (
        remaining
        .select_dtypes(include="number")
        .drop(columns=["target"], errors="ignore")
    )

    # Medians are robust to outliers and are learned from train only to avoid
    # using information from the test set.
    numeric_medians = numeric_data.median().to_dict()

    return {
        "dropped_cols": dropped_cols,
        "numeric_medians": numeric_medians,
        "categorical_fill_value": unknown_value
    }


def apply_missing_value_params(df, params):
    """Apply train-learned missing-value handling to any dataframe."""
    df = df.copy()

    df = df.drop(columns=params["dropped_cols"], errors="ignore")

    for col, median in params["numeric_medians"].items():
        if col in df.columns:
            df[col] = df[col].fillna(median)

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        # Fill categories with a visible token so models can learn whether
        # missingness itself is informative.
        df[col] = (
            df[col]
            .fillna(params["categorical_fill_value"])
            .astype(str)
        )

    return df


missing_params = fit_missing_value_params(train_processed)

train_processed = apply_missing_value_params(train_processed, missing_params)
test_processed = apply_missing_value_params(test_processed, missing_params)

print("dropped columns:")
print(missing_params["dropped_cols"])

print("train missing values:", train_processed.isna().sum().sum())
print("test missing values:", test_processed.isna().sum().sum())


## Time-Based Features

The creation time is useful because service speed can vary by hour, weekday, and weekend. Cyclic sine/cosine features keep times near midnight close to each other numerically.


In [ ]:
# 4) Derived temporal features.
# Turn date parts into shapes that are easier for models to learn.

def add_derived_temporal_features(df):
    """Add cyclic and binary time features from the created_* columns."""
    df = df.copy()

    # Sine/cosine encode repeating time cycles without creating artificial jumps
    # at boundaries such as 23:00 to 00:00 or Sunday to Monday.
    df["created_hour_sin"] = np.sin(2 * np.pi * df["created_hour"] / 24)
    df["created_hour_cos"] = np.cos(2 * np.pi * df["created_hour"] / 24)

    df["created_day_sin"] = np.sin(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_day_cos"] = np.cos(
        2 * np.pi * df["created_day_of_week"] / 7
    )

    df["created_month_sin"] = np.sin(
        2 * np.pi * df["created_month"] / 12
    )

    df["created_month_cos"] = np.cos(
        2 * np.pi * df["created_month"] / 12
    )

    # Binary indicators make common service patterns explicit.
    df["created_is_business_hour"] = (
        df["created_hour"].between(9, 17)
    ).astype(int)

    df["created_is_night"] = (
        df["created_hour"].between(0, 5)
    ).astype(int)

    return df


train_processed = add_derived_temporal_features(train_processed)
test_processed = add_derived_temporal_features(test_processed)

temporal_features = [
    "created_hour_sin",
    "created_hour_cos",
    "created_day_sin",
    "created_day_cos",
    "created_month_sin",
    "created_month_cos",
    "created_is_business_hour",
    "created_is_night"
]

print("temporal features added:", temporal_features)


## Geographic Features

Location can influence service speed. This step turns borough and coordinate information into cleaner geographic signals for later models.


In [ ]:
# 5) Geographic features.
# Convert location fields into simpler signals that models can use.

def add_geographic_features(df):
    """Add borough indicators, ZIP groups, and coordinate-based features."""
    df = df.copy()

    if "Borough" in df.columns:
        borough = df["Borough"].astype(str).str.upper()

        # One-hot borough flags give linear models direct access to each borough.
        df["is_manhattan"] = (borough == "MANHATTAN").astype(int)
        df["is_brooklyn"] = (borough == "BROOKLYN").astype(int)
        df["is_queens"] = (borough == "QUEENS").astype(int)
        df["is_bronx"] = (borough == "BRONX").astype(int)
        df["is_staten_island"] = (borough == "STATEN ISLAND").astype(int)

    if "Incident Zip" in df.columns:
        # Group ZIP codes by the first digits to keep a broader neighborhood-like
        # signal without treating every ZIP as unrelated.
        df["incident_zip_group"] = (
            df["Incident Zip"] // 100
        ).astype(int).astype(str)

    if "Latitude" in df.columns and "Longitude" in df.columns:
        nyc_latitude = 40.7128
        nyc_longitude = -74.0060

        # Approximate distance from the NYC center gives a compact location signal.
        df["distance_from_nyc_center"] = np.sqrt(
            (df["Latitude"] - nyc_latitude) ** 2
            + (df["Longitude"] - nyc_longitude) ** 2
        )

        # The coordinate interaction gives simple models another way to separate
        # geographic areas.
        df["latitude_longitude_interaction"] = (
            df["Latitude"] * df["Longitude"]
        )

    return df


train_processed = add_geographic_features(train_processed)
test_processed = add_geographic_features(test_processed)

geographic_features = [
    col
    for col in [
        "is_manhattan",
        "is_brooklyn",
        "is_queens",
        "is_bronx",
        "is_staten_island",
        "incident_zip_group",
        "distance_from_nyc_center",
        "latitude_longitude_interaction"
    ]
    if col in train_processed.columns
]

print("geographic features added:", geographic_features)


## Encoding Strategy

The preprocessing notebook converts messy tabular columns into model-ready numeric features. Count encoding is used for high-cardinality categories so the model can use category frequency without creating thousands of dummy columns.


In [ ]:
# 6) Count encoding.
# Count encodings convert high-cardinality categories into their training-set
# frequencies, which can be useful for models that need numeric inputs.

def fit_count_encoding(df, columns):
    """Learn category-frequency maps from the training dataframe only."""
    # Count encodings are fitted only on training data to avoid using test
    # distribution information.
    count_maps = {}

    for col in columns:
        if col in df.columns:
            count_maps[col] = (
                df[col]
                .astype(str)
                .value_counts(normalize=True)
                .to_dict()
            )

    return count_maps


def apply_count_encoding(df, count_maps):
    """Add one frequency feature for each fitted count-encoded column."""
    df = df.copy()

    for col, mapping in count_maps.items():
        if col in df.columns:
            # Unknown test categories receive 0 because they were never observed
            # in the training distribution used to build the map.
            df[f"{col}_frequency"] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(0)
            )

    return df


count_encoding_cols = [
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Borough",
    "Incident Zip",
    "incident_zip_group"
]

count_maps = fit_count_encoding(
    train_processed,
    count_encoding_cols
)

train_processed = apply_count_encoding(
    train_processed,
    count_maps
)

test_processed = apply_count_encoding(
    test_processed,
    count_maps
)

count_features = [
    f"{col}_frequency"
    for col in count_encoding_cols
    if f"{col}_frequency" in train_processed.columns
]

print("count-encoded features added:", count_features)


In [ ]:
# 7) Interaction features.
# Interaction features capture combinations that may be more informative than
# each column alone.

def add_interaction_features(df):
    """Add simple products between related engineered features."""
    df = df.copy()

    # Weekend-night requests may follow different operational patterns than
    # weekday daytime requests.
    df["weekend_night_interaction"] = (
        df["created_is_weekend"] * df["created_is_night"]
    )

    df["business_hour_weekday_interaction"] = (
        df["created_is_business_hour"] * (1 - df["created_is_weekend"])
    )

    if "Borough_frequency" in df.columns:
        # Combine borough frequency with weekend timing to capture whether common
        # boroughs behave differently on weekends.
        df["borough_weekend_interaction"] = (
            df["Borough_frequency"] * df["created_is_weekend"]
        )

    return df


train_processed = add_interaction_features(train_processed)
test_processed = add_interaction_features(test_processed)

interaction_features = [
    col
    for col in [
        "weekend_night_interaction",
        "business_hour_weekday_interaction",
        "borough_weekend_interaction"
    ]
    if col in train_processed.columns
]

print("interaction features added:", interaction_features)


In [ ]:
# 8) Categorical encoding.
# Remaining categorical columns must be converted to numbers before standard
# scikit-learn models can use them.

def fit_categorical_encoding_params(df):
    """Decide which categorical columns get one-hot vs label encoding."""
    categorical_cols = (
        df
        .select_dtypes(include=["object", "category"])
        .columns
        .tolist()
    )

    categorical_cols = [
        col
        for col in categorical_cols
        if col != "target"
    ]

    onehot_cols = []
    label_maps = {}

    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)

        if n_unique <= max_onehot_cardinality:
            # Low-cardinality categories are expanded into separate 0/1 columns.
            onehot_cols.append(col)
        else:
            # High-cardinality categories are label-encoded to avoid creating a
            # very wide feature matrix.
            categories = sorted(df[col].astype(str).unique().tolist())

            mapping = {
                category: index
                for index, category in enumerate(categories)
            }

            mapping["__UNKNOWN__"] = len(mapping)
            label_maps[col] = mapping

    return {
        "onehot_cols": onehot_cols,
        "label_maps": label_maps
    }


def apply_categorical_encoding(df, params):
    """Apply train-learned categorical encodings to train or test data."""
    df = df.copy()

    for col, mapping in params["label_maps"].items():
        if col in df.columns:
            unknown_code = mapping["__UNKNOWN__"]

            df[col] = (
                df[col]
                .astype(str)
                .map(mapping)
                .fillna(unknown_code)
                .astype(int)
            )

    existing_onehot_cols = [
        col
        for col in params["onehot_cols"]
        if col in df.columns
    ]

    # get_dummies expands only the low-cardinality columns selected during fit.
    df = pd.get_dummies(
        df,
        columns=existing_onehot_cols,
        dtype=int
    )

    return df


encoding_params = fit_categorical_encoding_params(train_processed)

train_encoded = apply_categorical_encoding(
    train_processed,
    encoding_params
)

test_encoded = apply_categorical_encoding(
    test_processed,
    encoding_params
)

print("one-hot columns:")
print(encoding_params["onehot_cols"])

print("label-encoded columns:")
print(list(encoding_params["label_maps"].keys()))

print("train shape after encoding:", train_encoded.shape)
print("test shape after encoding:", test_encoded.shape)


## Final Matrix Checks

The final checks make sure training and test matrices have the same feature columns. This prevents errors later when a model trained on one column order is asked to predict on another.


In [ ]:
# 9) Final matrices and checks.
# Create the objects that later notebooks expect: X_train, y_train,
# and X_test.

def create_final_matrices(train_df, test_df):
    """Return aligned train features, train target, and test features."""
    # Final contract: X_train and X_test must have identical
    # columns, while y_train contains the only target column.
    feature_cols = [
        col
        for col in train_df.columns
        if col != "target"
    ]

    X_train = train_df[feature_cols].copy()
    y_train = train_df["target"].copy()

    # Reindex test to the exact training feature order. Missing dummy columns are
    # filled with 0 because the category was absent from test.
    X_test = test_df.reindex(
        columns=feature_cols,
        fill_value=0
    ).copy()

    return X_train, y_train, X_test


X_train, y_train, X_test = create_final_matrices(
    train_encoded,
    test_encoded
)

# Catch preprocessing mistakes before modeling starts.
assert "target" not in X_train.columns
assert "target" not in X_test.columns
assert "Closed Date" not in X_train.columns
assert "Closed Date" not in X_test.columns
assert X_train.shape[1] == X_test.shape[1]
assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert len(X_train.select_dtypes(exclude="number").columns) == 0
assert len(X_test.select_dtypes(exclude="number").columns) == 0

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)


## Preprocessing Summary

The final summary explains what was created, removed, and kept. It is a compact audit trail for the preprocessing choices.


In [ ]:
# 10) Final preprocessing summary.
# Summarize the finished matrices and engineered features.
engineered_features = (
    temporal_features
    + geographic_features
    + count_features
    + interaction_features
)

summary = pd.DataFrame({
    "metric": [
        "training rows",
        "test rows",
        "final features",
        "engineered features tracked",
        "positive target rate",
        "missing values in X_train",
        "missing values in X_test",
        "non-numeric columns in X_train",
        "non-numeric columns in X_test"
    ],
    "value": [
        X_train.shape[0],
        X_test.shape[0],
        X_train.shape[1],
        len([col for col in engineered_features if col in train_processed.columns]),
        round(y_train.mean(), 4),
        int(X_train.isna().sum().sum()),
        int(X_test.isna().sum().sum()),
        len(X_train.select_dtypes(exclude="number").columns),
        len(X_test.select_dtypes(exclude="number").columns)
    ]
})

display(summary)

# Show the final target balance after preprocessing.
target_summary = (
    y_train
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(y_train) * 100
).round(2)

display(target_summary)

# Final quality check: columns with missing values or only a few unique values
# are worth reviewing before model training.
top_missing_or_constant_checks = pd.DataFrame({
    "missing_values": X_train.isna().sum(),
    "unique_values": X_train.nunique(),
    "dtype": X_train.dtypes.astype(str)
}).sort_values(["missing_values", "unique_values"], ascending=[False, True]).head(15)

display(top_missing_or_constant_checks)
